In [ ]:
import os

if not os.path.exists('/content/dataset'):
    !unzip "dataset_roboflow.zip" -d /content/dataset

In [ ]:
!pip install ultralytics

In [ ]:
%%writefile /content/dataset/data.yaml
train: ../train/images
val: ../train/images
# test: ../test/images # Commenting out test for now

nc: 1
names: ['bracket']

roboflow:
  workspace: -workspace-evmtz
  project: -workspace-evmtz
  version: dataset
  license: Private
  url: https://app.roboflow.com/-workspace-evmtz/-workspace-evmtz/dataset

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train2/weights/best.pt")

results = model.predict(
    source="/content/test_bracket_IO.jpeg",   # change this to your image path
    conf=0.25,
    save=True,
    imgsz=640
)

In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/runs/detect/predict/test_bracket_IO.jpg"))

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train2/weights/best.pt")

results = model.predict(
    source="/content/test_bracket_io_2.jpeg",   # change this to your image path
    conf=0.25,
    save=True,
    imgsz=640
)

In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/runs/detect/predict2/test_bracket_io_2.jpg"))

In [ ]:
metrics = model.val()

print("Precision:", metrics.box.p.mean())
print("Recall:", metrics.box.r.mean())
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
def calculate_iou(box1, box2):
    # box format: [x1, y1, x2, y2]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)

    box1_area = (box1[2]-box1[0]) * (box1[3]-box1[1])
    box2_area = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = box1_area + box2_area - inter_area

    return inter_area / union

In [ ]:
#this is just a trial to see if it is working - > above code works just fine

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO


def get_tooth_axis_angle(tooth_crop):
    gray = cv2.cvtColor(tooth_crop, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 30, 100)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=25,
                             minLineLength=15, maxLineGap=8)
    if lines is None:
        return 90.0
    angles = []
    for l in lines:
        x1, y1, x2, y2 = l[0]
        angle = np.degrees(np.arctan2(abs(y2 - y1), abs(x2 - x1)))
        if angle > 45:
            angles.append(angle)
    if not angles:
        return 90.0
    return float(np.median(angles))


def classify_placement(horiz_off, vert_off, tilt):
    if horiz_off < 8 and vert_off < 10 and tilt < 5:
        return "Correct", "#2ecc71"
    elif horiz_off < 20 and vert_off < 22 and tilt < 15:
        return "Slight deviation", "#f39c12"
    else:
        return "Misplaced", "#e74c3c"


def analyze_bracket_placement(image_path, model, conf=0.25):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    H, W = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    results = model.predict(source=image_path, conf=conf, save=False,
                            imgsz=640, verbose=False)
    detections = []

    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        bw, bh = x2 - x1, y2 - y1
        bracket_cx = (x1 + x2) / 2
        bracket_cy = (y1 + y2) / 2

        pad_x = int(bw * 0.8)
        pad_y = int(bh * 1.5)
        tx1 = max(0, x1 - pad_x)
        ty1 = max(0, y1 - pad_y)
        tx2 = min(W, x2 + pad_x)
        ty2 = min(H, y2 + pad_y)
        tooth_crop = img_bgr[ty1:ty2, tx1:tx2]

        rel_cx = (bracket_cx - tx1) / (tx2 - tx1)
        rel_cy = (bracket_cy - ty1) / (ty2 - ty1)

        horiz_offset = abs(rel_cx - 0.5) * 100
        vert_offset  = abs(rel_cy - 0.5) * 100

        tooth_angle  = get_tooth_axis_angle(tooth_crop)
        bracket_tilt = abs(tooth_angle - 90.0)

        status, color = classify_placement(horiz_offset, vert_offset, bracket_tilt)

        detections.append({
            "bbox"         : [x1, y1, x2, y2],
            "tooth_roi"    : [tx1, ty1, tx2, ty2],
            "horiz_off_%"  : round(horiz_offset, 1),
            "vert_off_%"   : round(vert_offset, 1),
            "tilt_deg"     : round(bracket_tilt, 1),
            "status"       : status,
            "color"        : color,
        })

    return img_rgb, detections

In [ ]:
def visualize_results(img_rgb, detections, figsize=(12, 8)):
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img_rgb)
    ax.axis("off")

    for i, d in enumerate(detections):
        x1, y1, x2, y2     = d["bbox"]
        tx1, ty1, tx2, ty2 = d["tooth_roi"]
        status = d["status"]
        color  = d["color"]

        ax.add_patch(patches.Rectangle(
            (tx1, ty1), tx2-tx1, ty2-ty1,
            linewidth=1, edgecolor="white", facecolor="none",
            linestyle="--", alpha=0.5))

        ax.add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=color, facecolor=color, alpha=0.15))
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=color, facecolor="none"))

        label = (f"#{i+1} {status}\n"
                 f"H:{d['horiz_off_%']}%  "
                 f"V:{d['vert_off_%']}%  "
                 f"T:{d['tilt_deg']}°")
        card_y = max(y1 - 8, 10)
        ax.text(x1, card_y, label, fontsize=7.5, color="white", va="bottom",
                bbox=dict(boxstyle="round,pad=0.35",
                          fc=color, alpha=0.88, ec="none"))

    ax.set_title(
        f"Bracket placement analysis  —  {len(detections)} bracket(s) detected",
        fontsize=11, pad=10)
    plt.tight_layout()
    plt.show()

    print(f"\n{'#':<4} {'Status':<20} {'H offset':>10} {'V offset':>10} {'Tilt':>8}")
    print("-" * 56)
    for i, d in enumerate(detections):
        print(f"{i+1:<4} {d['status']:<20} "
              f"{d['horiz_off_%']:>9.1f}%"
              f"{d['vert_off_%']:>9.1f}%"
              f"{d['tilt_deg']:>7.1f}°")

In [ ]:
# Load the trained model
model = YOLO("/content/best.pt")

# --- Test image 1 ---
img, dets = analyze_bracket_placement("/content/test_bracket_IO.jpeg", model, conf=0.25)
visualize_results(img, dets)

# --- Test image 2 ---
img2, dets2 = analyze_bracket_placement("/content/test_bracket_io_2.jpeg", model, conf=0.25)
visualize_results(img2, dets2)